# Topic 4: Spark UI Mastery

> **Phase 4 → Week 7 → Topic 4**
>
> Prerequisites: Topics 1-3 (Shuffle, Skew, AQE)

---

## Why the Spark UI Matters

The Spark UI is your primary diagnostic tool. Every performance problem you've learned about — skew, spill, too many shuffles, GC pressure — has a fingerprint in the Spark UI. Knowing where to look and what the numbers mean turns you from a "guess and check" engineer into a systematic performance engineer.

---

## UI Navigation Map

```
Spark UI (http://localhost:4040)
│
├── Jobs        ← High level: what jobs ran, how long
│   └── Click a job → see its stages
│
├── Stages      ← Medium level: stages, shuffle metrics, spill
│   └── Click a stage → see individual tasks
│
├── Tasks       ← Fine level: each task's duration, input, GC
│   (in stage detail)
│
├── Storage     ← What is cached and at what storage level
│
├── Environment ← SparkConf settings actually in effect
│
├── Executors   ← Per-executor memory, GC, active tasks
│
└── SQL         ← DataFrame/SQL query plans, stage DAGs
    └── Click a query → see the physical plan + stage DAG
```

---

## Interview Cheat Sheet

**Q: How do you use the Spark UI to diagnose a slow job?**
> 1. Jobs tab → find the slow job, note its stage IDs. 2. Stages tab → find the stage with the longest duration. 3. Click the stage → Task Metrics → look for: (a) one task much longer than others (skew — check input size column), (b) Spill (Disk) > 0 (need more partitions or memory), (c) GC Time > 10% of task time (GC pressure). 4. SQL tab → find the query → check DAG for unexpected Exchange (shuffle) nodes or large data sizes.

**Q: What is a stage in Spark and how is it visible in the UI?**
> A stage is a set of tasks that can run in parallel without a shuffle boundary. Each Exchange (shuffle) in the physical plan creates a new stage. In the UI, Stages tab shows one row per stage with duration, task counts, shuffle read/write, and input size. A healthy stage has consistent task durations; a skewed stage has one outlier task.

**Q: What metrics in the Stages tab indicate problems?**
> Spill (Memory/Disk) > 0: memory pressure, fix with more partitions or more memory. One task with 10x more input size than others: data skew, fix with salting or AQE. GC Time > 10% of task time: GC pressure, fix with G1GC or MEMORY_ONLY_SER. Shuffle Read Size >> Shuffle Write Size: data is expanding during the shuffle (e.g., explode or join that creates more rows than input).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = SparkSession.builder \
    .appName("Week7 - Spark UI Mastery") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("Spark UI available at: http://localhost:4040")
print()
print("This notebook generates specific jobs designed to show")
print("identifiable patterns in the Spark UI.")
print("Keep the UI open in a browser tab while running each cell.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/16 08:08:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI available at: http://localhost:4040

This notebook generates specific jobs designed to show
identifiable patterns in the Spark UI.
Keep the UI open in a browser tab while running each cell.


## Part 1: Jobs Tab — Understanding Job Lifecycle

In [2]:
# Each action creates one job
# Let's create distinct named jobs so they're easy to find in the UI

df = spark.range(1_000_000) \
    .withColumn("key", (F.col("id") % 100).cast("string")) \
    .withColumn("value", F.rand() * 1000)

# Job 1: simple count (1 stage, no shuffle)
spark.sparkContext.setJobDescription("Job1: Simple count (narrow, no shuffle)")
count1 = df.count()
print(f"Job 1 — count: {count1:,}")

# Job 2: filter + count (1 stage, narrow transformation)
spark.sparkContext.setJobDescription("Job2: Filter + count (narrow, no shuffle)")
count2 = df.filter(F.col("value") > 500).count()
print(f"Job 2 — filtered count: {count2:,}")

# Job 3: groupBy (2 stages: map stage + reduce stage, 1 shuffle)
spark.sparkContext.setJobDescription("Job3: GroupBy (wide, 1 shuffle)")
grouped = df.groupBy("key").agg(F.sum("value").alias("total"), F.count("*").alias("cnt"))
grouped.count()
print("Job 3 — groupBy complete")

print()
print("Go to UI → Jobs tab:")
print("  Job 1, 2: 1 stage each (no Exchange in plan)")
print("  Job 3:    2 stages (1 Exchange → stage boundary)")
print()
print("Each job shows: status, duration, stages, tasks, input/output")

Job 1 — count: 1,000,000


Job 2 — filtered count: 499,960


Job 3 — groupBy complete

Go to UI → Jobs tab:
  Job 1, 2: 1 stage each (no Exchange in plan)
  Job 3:    2 stages (1 Exchange → stage boundary)

Each job shows: status, duration, stages, tasks, input/output


## Part 2: Stages Tab — Reading Stage Metrics

In [3]:
print("""
Stages Tab — Column Reference:
════════════════════════════════════════════════════════════════
Column               What it means
───────────────────  ──────────────────────────────────────────────
Stage Id             Sequential ID, higher = later
Description          From setJobDescription() or auto-generated
Submitted            When stage was submitted to scheduler
Duration             Wall clock time from first task start to last task end
Tasks                Completed / Total (e.g. 8/8)
Input                Data read from HDFS/disk (only for first stage)
Output               Data written to HDFS/disk (only for save stages)
Shuffle Read         Total bytes read from shuffle files (network I/O)
Shuffle Write        Total bytes written to shuffle files (disk I/O)
Spill (Memory)       Data that was in memory before spilling (pre-spill size)
Spill (Disk)         Compressed bytes written to disk during spill

Quick triage:
  Spill (Disk) > 0          → increase partitions or memory
  Shuffle Read >> Write     → join or explode creating more rows
  Duration >> other stages  → likely has skew or spill — click in
════════════════════════════════════════════════════════════════
""")

# Create a job with spill-like characteristics (large aggregation)
spark.sparkContext.setJobDescription("Demo: Large aggregation — check Shuffle Write")
large_agg = spark.range(2_000_000) \
    .withColumn("key", (F.col("id") % 8).cast("string")) \
    .withColumn("data", F.repeat(F.lit("x"), 20)) \
    .withColumn("value", F.rand() * 1000) \
    .groupBy("key") \
    .agg(F.sum("value"), F.count("*"))

large_agg.count()
print("Large aggregation complete — check Stages tab for Shuffle Write size")


Stages Tab — Column Reference:
════════════════════════════════════════════════════════════════
Column               What it means
───────────────────  ──────────────────────────────────────────────
Stage Id             Sequential ID, higher = later
Description          From setJobDescription() or auto-generated
Submitted            When stage was submitted to scheduler
Duration             Wall clock time from first task start to last task end
Tasks                Completed / Total (e.g. 8/8)
Input                Data read from HDFS/disk (only for first stage)
Output               Data written to HDFS/disk (only for save stages)
Shuffle Read         Total bytes read from shuffle files (network I/O)
Shuffle Write        Total bytes written to shuffle files (disk I/O)
Spill (Memory)       Data that was in memory before spilling (pre-spill size)
Spill (Disk)         Compressed bytes written to disk during spill

Quick triage:
  Spill (Disk) > 0          → increase partitions or memory
 

Large aggregation complete — check Stages tab for Shuffle Write size


## Part 3: Stage Detail — Task-Level Metrics

In [4]:
print("""
Stage Detail — Tasks Tab Column Reference:
════════════════════════════════════════════════════════════════
Column               What it means
───────────────────  ──────────────────────────────────────────────
Index                Task ID within the stage
Attempt              Usually 0; >0 means task was retried
Status               SUCCESS / FAILED / RUNNING / KILLED
Locality             PROCESS_LOCAL / NODE_LOCAL / RACK_LOCAL / ANY
                     (PROCESS_LOCAL = data in executor memory cache, fastest)
Executor             Which executor ran this task
Launch Time          Wall clock start time
Duration             Task wall clock time

# Time Breakdown (expand task row):
Scheduler Delay      Time waiting in queue before task started
Task Deserialization Time to deserialize task from driver
GC Time              Time spent in garbage collection (alert: >10% of Duration)
Result Serialization Time to serialize result back to driver
Getting Result Time  Time driver waited to receive the result

# Data Metrics:
Input Size / Records Data read from storage (first stage only)
Shuffle Read Size    Data read from shuffle files (network)
Shuffle Read Blocked Time waiting for shuffle data to arrive
Shuffle Write Size   Data written to shuffle files
Spill (Memory)       Data in memory before spill
Spill (Disk)         Bytes written to disk during spill

What to look for:
  One task Duration >> others: skew → check Input Size or Shuffle Read Size
  GC Time > 10% of Duration: memory pressure → MEMORY_ONLY_SER, G1GC
  Attempt > 0: task failures → check executor logs
  Locality = ANY: data not local → check storage/network setup
  Scheduler Delay >> Compute: overloaded driver or too many partitions
════════════════════════════════════════════════════════════════
""")


Stage Detail — Tasks Tab Column Reference:
════════════════════════════════════════════════════════════════
Column               What it means
───────────────────  ──────────────────────────────────────────────
Index                Task ID within the stage
Attempt              Usually 0; >0 means task was retried
Status               SUCCESS / FAILED / RUNNING / KILLED
Locality             PROCESS_LOCAL / NODE_LOCAL / RACK_LOCAL / ANY
                     (PROCESS_LOCAL = data in executor memory cache, fastest)
Executor             Which executor ran this task
Launch Time          Wall clock start time
Duration             Task wall clock time

# Time Breakdown (expand task row):
Scheduler Delay      Time waiting in queue before task started
Task Deserialization Time to deserialize task from driver
GC Time              Time spent in garbage collection (alert: >10% of Duration)
Result Serialization Time to serialize result back to driver
Getting Result Time  Time driver waited to recei

In [5]:
# Generate a skewed job — easily visible in UI
spark.sparkContext.setJobDescription("SKEW DEMO — look for outlier task in Stage Detail")

skewed_df = spark.createDataFrame(
    [("HOT", i, float(i)) for i in range(200_000)] +  # hot key
    [("B",   i, float(i)) for i in range(500)] +
    [("C",   i, float(i)) for i in range(500)] +
    [("D",   i, float(i)) for i in range(500)] +
    [("E",   i, float(i)) for i in range(500)],
    ["customer_id", "order_id", "amount"]
)

skewed_result = skewed_df.groupBy("customer_id") \
    .agg(F.sum("amount").alias("total"), F.count("*").alias("cnt"))

t0 = time.time()
skewed_result.show()
print(f"Completed in {time.time()-t0:.2f}s")

print()
print("Go to: Stages tab → click the second stage (shuffle read stage)")
print("        → Tasks tab → sort by 'Duration' descending")
print("You will see one task with far more data and longer duration (HOT key)")

+-----------+----------+------+
|customer_id|     total|   cnt|
+-----------+----------+------+
|          B|  124750.0|   500|
|          C|  124750.0|   500|
|          E|  124750.0|   500|
|          D|  124750.0|   500|
|        HOT|1.99999E10|200000|
+-----------+----------+------+

Completed in 2.06s

Go to: Stages tab → click the second stage (shuffle read stage)
        → Tasks tab → sort by 'Duration' descending
You will see one task with far more data and longer duration (HOT key)


## Part 4: SQL Tab — DAG and Plan Reading

In [6]:
# The SQL tab shows the DAG (Directed Acyclic Graph) of query execution
# Each node is one physical operator; colors show which stage it belongs to

print("""
SQL Tab — DAG Visualization:
════════════════════════════════════════════════════════════════
Each box in the DAG = one physical operator:

  FileScan parquet       → reading from storage
  Filter                 → predicate pushdown (good — happens early)
  Project                → column pruning (good — drops unused columns)
  Exchange (shuffle)     → ← STAGE BOUNDARY, most expensive
  HashAggregate          → local aggregation (partial, before shuffle)
  Sort                   → sorting within a partition
  SortMergeJoin          → join after both sides sorted (2 shuffles)
  BroadcastHashJoin      → join with broadcast (0 shuffles on the large side)
  BroadcastExchange      → sending small table to all executors

Colors:
  Each stage has a color band — adjacent operators with same color = same stage
  Exchange node = color break → new stage begins

Numbers on each node:
  'output: 1,234,567 rows' → actual row count for that operator
  'time: 1.2 s'            → wall time for that operator
  'spill: 45.0 MiB'        → spill for this operator

What to look for:
  Many Exchange nodes = many shuffles = expensive
  SortMergeJoin on large tables = can it be BroadcastHashJoin?
  Filter AFTER Exchange = predicate not pushed down (should be before)
  Large 'output rows' near the end = row explosion from explode/join
════════════════════════════════════════════════════════════════
""")

# Generate a complex query with multiple stages
spark.sparkContext.setJobDescription("Complex query — 3 shuffles visible in SQL DAG")

orders   = spark.read.csv("/workspace/week4/data/orders.csv",   header=True, inferSchema=True)
products = spark.read.csv("/workspace/week4/data/products.csv", header=True, inferSchema=True)

complex_query = (
    orders
    .join(products, on="product_id", how="inner")   # shuffle 1 (orders) + shuffle 2 (products)
    .groupBy("category")                             # shuffle 3
    .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders"))
    .orderBy(F.col("revenue").desc())
)
complex_query.show()

print("Now check SQL tab → click the query → examine the DAG")
print("Count the Exchange nodes — each is a stage boundary")


SQL Tab — DAG Visualization:
════════════════════════════════════════════════════════════════
Each box in the DAG = one physical operator:

  FileScan parquet       → reading from storage
  Filter                 → predicate pushdown (good — happens early)
  Project                → column pruning (good — drops unused columns)
  Exchange (shuffle)     → ← STAGE BOUNDARY, most expensive
  HashAggregate          → local aggregation (partial, before shuffle)
  Sort                   → sorting within a partition
  SortMergeJoin          → join after both sides sorted (2 shuffles)
  BroadcastHashJoin      → join with broadcast (0 shuffles on the large side)
  BroadcastExchange      → sending small table to all executors

Colors:
  Each stage has a color band — adjacent operators with same color = same stage
  Exchange node = color break → new stage begins

Numbers on each node:
  'output: 1,234,567 rows' → actual row count for that operator
  'time: 1.2 s'            → wall time for that o

+-----------+------------------+------+
|   category|           revenue|orders|
+-----------+------------------+------+
|Electronics| 7349.799999999997|    11|
|  Furniture|           1469.94|     4|
|      Books|359.90999999999997|     5|
+-----------+------------------+------+

Now check SQL tab → click the query → examine the DAG
Count the Exchange nodes — each is a stage boundary


## Part 5: Executors Tab

In [7]:
print("""
Executors Tab — Column Reference:
════════════════════════════════════════════════════════════════
Column               What it means
───────────────────  ──────────────────────────────────────────────
Executor ID          driver=0, workers start at 1
Address              Host:port of the executor JVM
Status              Active / Dead (Dead = executor crashed)
RDD Blocks           # of cached RDD/DataFrame partitions on this executor
Storage Memory       Used / Total memory for caching
Disk Used            Shuffle and spill files on this executor's local disk
Cores                # of task slots (= executor.cores)
Active Tasks         Currently running tasks
Failed Tasks         Total failed task attempts (alert: > 0 needs investigation)
Completed Tasks      Cumulative completed tasks
Total GC Time        Cumulative GC time (alert: > 10% of total Duration)
Total Duration       Cumulative task CPU time
Shuffle Read         Total bytes read from shuffle by this executor
Shuffle Write        Total bytes written to shuffle by this executor

Diagnostic patterns:
  Failed Tasks > 0:        Executor is struggling — check logs
  Dead executor:           OOM, node failure, or GC pause too long
  GC Time >> Compute:      GC pressure — reduce heap objects, use G1GC
  One executor >> others:  Data/compute imbalance
  Storage Memory near 100%: Cache filling up — may cause evictions
════════════════════════════════════════════════════════════════
""")


Executors Tab — Column Reference:
════════════════════════════════════════════════════════════════
Column               What it means
───────────────────  ──────────────────────────────────────────────
Executor ID          driver=0, workers start at 1
Address              Host:port of the executor JVM
Status              Active / Dead (Dead = executor crashed)
RDD Blocks           # of cached RDD/DataFrame partitions on this executor
Storage Memory       Used / Total memory for caching
Disk Used            Shuffle and spill files on this executor's local disk
Cores                # of task slots (= executor.cores)
Active Tasks         Currently running tasks
Failed Tasks         Total failed task attempts (alert: > 0 needs investigation)
Completed Tasks      Cumulative completed tasks
Total GC Time        Cumulative GC time (alert: > 10% of total Duration)
Total Duration       Cumulative task CPU time
Shuffle Read         Total bytes read from shuffle by this executor
Shuffle Write   

## Part 6: Storage Tab

In [8]:
# Generate cached data to see in Storage tab
from pyspark import StorageLevel

orders = spark.read.csv("/workspace/week4/data/orders.csv", header=True, inferSchema=True)
orders.persist(StorageLevel.MEMORY_AND_DISK)
orders.count()  # trigger caching

print("Orders cached — check Storage tab for:")
print("""
Storage Tab — Column Reference:
════════════════════════════════════════════════════════════════
RDD Name             DataFrame/RDD name (or auto-generated)
Storage Level        MEMORY_ONLY / MEMORY_AND_DISK / DISK_ONLY etc.
Cached Partitions    # of partitions currently in cache
Fraction Cached      % of total partitions in cache
Size in Memory       Total bytes of in-memory cached data
Size on Disk         Total bytes spilled to disk (if MEMORY_AND_DISK)

What to look for:
  Fraction Cached < 100%: cache doesn't fit — partitions evicted by execution
  Size on Disk > 0:       memory full, some partitions spilled to disk
  Many entries:           unpersist DataFrames you no longer need!

When to cache:
  ✓ DataFrame accessed 2+ times in different queries
  ✓ Expensive computation (join + agg) that feeds multiple paths
  ✗ Single-use DataFrames (cache overhead outweighs benefit)
  ✗ Data larger than available memory (evictions will slow everything)
════════════════════════════════════════════════════════════════
""")

# Use the cached data
spark.sparkContext.setJobDescription("Using cached orders")
orders.groupBy("status").count().show()
orders.groupBy("product_id").agg(F.sum("amount")).show(5)

orders.unpersist()
print("\nOrders unpersisted — Storage tab should now be empty")

Orders cached — check Storage tab for:

Storage Tab — Column Reference:
════════════════════════════════════════════════════════════════
RDD Name             DataFrame/RDD name (or auto-generated)
Storage Level        MEMORY_ONLY / MEMORY_AND_DISK / DISK_ONLY etc.
Cached Partitions    # of partitions currently in cache
Fraction Cached      % of total partitions in cache
Size in Memory       Total bytes of in-memory cached data
Size on Disk         Total bytes spilled to disk (if MEMORY_AND_DISK)

What to look for:
  Fraction Cached < 100%: cache doesn't fit — partitions evicted by execution
  Size on Disk > 0:       memory full, some partitions spilled to disk
  Many entries:           unpersist DataFrames you no longer need!

When to cache:
  ✓ DataFrame accessed 2+ times in different queries
  ✓ Expensive computation (join + agg) that feeds multiple paths
  ✗ Single-use DataFrames (cache overhead outweighs benefit)
  ✗ Data larger than available memory (evictions will slow everything

+----------+-----+
|    status|count|
+----------+-----+
|processing|    2|
| delivered|   15|
|   shipped|    2|
| cancelled|    1|
+----------+-----+



+----------+-----------+
|product_id|sum(amount)|
+----------+-----------+
|      P004|     899.97|
|      P009|     119.98|
|      P001|    6499.95|
|      P008|     399.98|
|      P002|      299.9|
+----------+-----------+
only showing top 5 rows


Orders unpersisted — Storage tab should now be empty


## Part 7: Environment Tab

In [9]:
print("""
Environment Tab:
════════════════════════════════════════════════════════════════
Shows all SparkConf settings actually in effect.
Critical for debugging configuration issues:

  Q: Is AQE actually enabled?  → Search 'adaptive.enabled'
  Q: What's shuffle.partitions set to?  → Search 'shuffle.partitions'
  Q: Is G1GC enabled?  → Look in 'executor.extraJavaOptions'
  Q: Is dynamic allocation on?  → Search 'dynamicAllocation'

Also shows:
  Spark Properties: all spark.* configs
  Hadoop Properties: HDFS configs
  System Properties: Java system properties
  Classpath Entries: JARs on the classpath

Tip: When a config 'isn't working', always check here first.
     The setting might not be applied the way you think.
════════════════════════════════════════════════════════════════
""")

# Print current config programmatically
print("Current Spark configuration (key settings):")
configs = [
    "spark.sql.adaptive.enabled",
    "spark.sql.shuffle.partitions",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.executor.memory",
    "spark.driver.memory",
]
for cfg in configs:
    try:
        val = spark.conf.get(cfg)
        print(f"  {cfg} = {val}")
    except Exception:
        print(f"  {cfg} = (not set, using default)")


Environment Tab:
════════════════════════════════════════════════════════════════
Shows all SparkConf settings actually in effect.
Critical for debugging configuration issues:

  Q: Is AQE actually enabled?  → Search 'adaptive.enabled'
  Q: What's shuffle.partitions set to?  → Search 'shuffle.partitions'
  Q: Is G1GC enabled?  → Look in 'executor.extraJavaOptions'
  Q: Is dynamic allocation on?  → Search 'dynamicAllocation'

Also shows:
  Spark Properties: all spark.* configs
  Hadoop Properties: HDFS configs
  System Properties: Java system properties
  Classpath Entries: JARs on the classpath

Tip: When a config 'isn't working', always check here first.
     The setting might not be applied the way you think.
════════════════════════════════════════════════════════════════

Current Spark configuration (key settings):
  spark.sql.adaptive.enabled = false
  spark.sql.shuffle.partitions = 8
  spark.sql.autoBroadcastJoinThreshold = 10485760b


  spark.executor.memory = (not set, using default)


  spark.driver.memory = (not set, using default)


## Part 8: Systematic Diagnosis Workflow

In [10]:
print("""
Systematic Spark UI Diagnosis Workflow:
═══════════════════════════════════════════════════════════════════
SYMPTOM: Job is slow

Step 1 → Jobs Tab
  □ Find the job. Which stage takes the most time?
  □ How many stages? (= number of shuffles + 1)
    → Too many stages? Look for avoidable shuffles.

Step 2 → Stages Tab
  □ Open the slow stage.
  □ Check Spill (Disk): if > 0 → partition count too low
  □ Check Shuffle Write size: if huge → may need repartition/filter earlier

Step 3 → Stage Detail (Tasks)
  □ Sort by Duration descending.
  □ Is task 0 >> all others? → SKEW
     - Check Input Size / Shuffle Read Size for that task
     - Fix: AQE skew join or salting
  □ Are all tasks similarly slow? → NOT skew, might be resource contention
  □ Is GC Time > 10% of Duration? → GC pressure
     - Fix: G1GC, MEMORY_ONLY_SER, reduce caching
  □ Attempt > 0 on any task? → Task failures, check executor logs

Step 4 → SQL Tab
  □ Find the query. Count Exchange nodes.
  □ Any SortMergeJoin that should be BroadcastHashJoin?
  □ Is Filter before or after Exchange? (before = good, after = bad)
  □ Row count explosion anywhere? (explode, cross join)

Step 5 → Executors Tab
  □ Any dead executors? → OOM, node failure
  □ GC Time >> others? → One executor overloaded
  □ Storage Memory near 100%? → Evictions happening

COMMON FIXES MAPPED TO SYMPTOMS:
  Spill visible           → increase shuffle.partitions
  One slow task (skew)    → AQE skew join or salting
  Many Exchange nodes     → broadcast small tables, combine operations
  GC pressure             → G1GC + MEMORY_ONLY_SER
  Dead executors          → OOM diagnosis workflow (see Week 6 Topic 4)
═══════════════════════════════════════════════════════════════════
""")


Systematic Spark UI Diagnosis Workflow:
═══════════════════════════════════════════════════════════════════
SYMPTOM: Job is slow

Step 1 → Jobs Tab
  □ Find the job. Which stage takes the most time?
  □ How many stages? (= number of shuffles + 1)
    → Too many stages? Look for avoidable shuffles.

Step 2 → Stages Tab
  □ Open the slow stage.
  □ Check Spill (Disk): if > 0 → partition count too low
  □ Check Shuffle Write size: if huge → may need repartition/filter earlier

Step 3 → Stage Detail (Tasks)
  □ Sort by Duration descending.
  □ Is task 0 >> all others? → SKEW
     - Check Input Size / Shuffle Read Size for that task
     - Fix: AQE skew join or salting
  □ Are all tasks similarly slow? → NOT skew, might be resource contention
  □ Is GC Time > 10% of Duration? → GC pressure
     - Fix: G1GC, MEMORY_ONLY_SER, reduce caching
  □ Attempt > 0 on any task? → Task failures, check executor logs

Step 4 → SQL Tab
  □ Find the query. Count Exchange nodes.
  □ Any SortMergeJoin that 

## Exercises

1. Run the complex query (join + groupBy + orderBy on orders + products). In the SQL tab, count the Exchange nodes and explain why each shuffle is necessary.
2. Create a deliberately skewed groupBy (one key = 90% of data). Open Stage Detail and identify which task shows the skew. What column gives it away?
3. Cache a DataFrame, run two queries on it. In the Storage tab, find the cached entry. What is the storage level shown?
4. In the Environment tab (or programmatically), verify whether AQE is enabled. If it is, which three sub-features are also active?
5. A stage shows: Total Duration=120s, GC Time=15s, Input=2GB, Spill(Disk)=500MB. List ALL the problems and a fix for each.

In [11]:
# Exercise 2: Skewed groupBy — examine in UI
spark.sparkContext.setJobDescription("Exercise 2: Skewed groupBy — find outlier task")

spark.conf.set("spark.sql.adaptive.enabled", "false")  # disable AQE to see raw skew

ex_skewed = spark.createDataFrame(
    [("HOT_KEY", float(i)) for i in range(200_000)] +
    [("KEY_B", float(i)) for i in range(1_000)] +
    [("KEY_C", float(i)) for i in range(1_000)] +
    [("KEY_D", float(i)) for i in range(1_000)],
    ["key", "value"]
)

ex_skewed.groupBy("key") \
         .agg(F.sum("value"), F.count("*")) \
         .show()

print("Go to Stages → click the second stage → Tasks tab")
print("Sort by 'Duration' or 'Shuffle Read Size' descending")
print("The HOT_KEY task will show 60x more data than others")

+-------+----------+--------+
|    key|sum(value)|count(1)|
+-------+----------+--------+
|  KEY_C|  499500.0|    1000|
|HOT_KEY|1.99999E10|  200000|
|  KEY_D|  499500.0|    1000|
|  KEY_B|  499500.0|    1000|
+-------+----------+--------+

Go to Stages → click the second stage → Tasks tab
Sort by 'Duration' or 'Shuffle Read Size' descending
The HOT_KEY task will show 60x more data than others


In [12]:
# Exercise 5: Analyzing stage metrics
print("""
Exercise 5 Analysis:
Stage: Total Duration=120s, GC Time=15s, Input=2GB, Spill(Disk)=500MB

Problem 1: GC Time = 15s / 120s = 12.5% of duration
  Rule: GC > 10% = problem
  Fix: Enable G1GC (-XX:+UseG1GC), use MEMORY_ONLY_SER for caching,
       reduce heap objects (avoid Python UDFs with large objects)

Problem 2: Spill(Disk) = 500MB > 0
  Means: execution memory was exhausted mid-operation
  Fix: Increase spark.sql.shuffle.partitions (smaller data per partition)
       OR increase executor memory

Problem 3: Input = 2GB for a shuffle stage (if Shuffle Read = 2GB)
  This might indicate data wasn't filtered before the shuffle
  Fix: Add filters/projections before the shuffle-triggering operation

Cheapest fix order:
  1. Increase shuffle.partitions (free — reduces spill)
  2. Enable G1GC (-XX:+UseG1GC, free — reduces GC)
  3. Add pre-shuffle filters (code change — reduces input)
  4. Increase executor memory (costs resources)
""")


Exercise 5 Analysis:
Stage: Total Duration=120s, GC Time=15s, Input=2GB, Spill(Disk)=500MB

Problem 1: GC Time = 15s / 120s = 12.5% of duration
  Rule: GC > 10% = problem
  Fix: Enable G1GC (-XX:+UseG1GC), use MEMORY_ONLY_SER for caching,
       reduce heap objects (avoid Python UDFs with large objects)

Problem 2: Spill(Disk) = 500MB > 0
  Means: execution memory was exhausted mid-operation
  Fix: Increase spark.sql.shuffle.partitions (smaller data per partition)
       OR increase executor memory

Problem 3: Input = 2GB for a shuffle stage (if Shuffle Read = 2GB)
  This might indicate data wasn't filtered before the shuffle
  Fix: Add filters/projections before the shuffle-triggering operation

Cheapest fix order:
  1. Increase shuffle.partitions (free — reduces spill)
  2. Enable G1GC (-XX:+UseG1GC, free — reduces GC)
  3. Add pre-shuffle filters (code change — reduces input)
  4. Increase executor memory (costs resources)

